In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import pandas as pd
import os
import random
from torch.utils.data import random_split
import re
from collections import Counter, OrderedDict
from typing import Dict, List

**Project-One - Predicting the sentiment of IMDB movie reviews:**

In [2]:
a = "/users/blaise"
b = "train"
os.path.join(a,b)

'/users/blaise/train'

In [3]:
class TextDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.samples = []
        self.transform = transform

        combined_path = os.path.join(root_dir, split)

        for label_name in ['pos','neg']:
            label_dir = os.path.join(combined_path, label_name)
            label = 1 if label_name == 'pos' else 0
            for fname in os.listdir(label_dir):
                if fname.endswith('.txt'):
                    path = os.path.join(label_dir, fname)
                    self.samples.append((path, label))
        random.shuffle(self.samples)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with open(path, 'r', encoding ='utf-8') as f:
            text = f.read()
        if self.transform:
            text = self.transform(text)
        return text, label

In [4]:
root_dir = "/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/data/aclImdb"
train_dataset = TextDataset(root_dir=root_dir, split='train')
test_dataset = TextDataset(root_dir=root_dir, split='test')

In [5]:
len(train_dataset), len(test_dataset)

(25000, 25000)

In [9]:
train_dataset[0]

('This film sat on my Tivo for weeks before I watched it. I dreaded a self-indulgent yuppie flick about relationships gone bad. I was wrong; this was an engrossing excursion into the screwed-up libidos of New Yorkers.<br /><br />The format is the same as Max Ophuls\' "La Ronde," based on a play by Arthur Schnitzler, who is given an "inspired by" credit. It starts from one person, a prostitute , standing on a street corner in Brooklyn. She is picked up by a home contractor, who has sex with her on the hood of a car, but can\'t come. He refuses to pay her. When he\'s off peeing, she answers his cell phone and takes a message. She runs away with his keys.<br /><br />Then the story switches to the contractor, who pays a professional call on a rich, bored New York woman, who plays with him until he is aroused, then she pulls away. She tells him how desperate and unhappy she is; he tells her how beautiful she is, and lucky. As he is leaving, she asks if he would have sex with her. She sits o

Creating a validation set from the training set:

In [10]:
train_dataset, valid_dataset = random_split(train_dataset, [20000, 5000])

In [11]:
len(train_dataset), len(valid_dataset)

(20000, 5000)

To prepare data for input to a NN, we need to encode it to numeric values. we need to find the unique words (tokens) in the training dataset. we will use the python counter class to find the unique tokens in the text. we use a tokenizer function here, which, also cleans up th etext, removing html markups as well as punctuation and other non-letter characters

In [12]:
def tokenizer(text):
    text = re.sub('<[^>]*>','', text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
    text = re.sub("[^\w']+",' ', text.lower()) + ' ' +' '.join(emoticons).replace('-','')
    tokenized = text.split()
    return tokenized

<>:3: SyntaxWarning: invalid escape sequence '\)'
<>:4: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\)'
<>:4: SyntaxWarning: invalid escape sequence '\w'
/var/folders/_h/yzv4_kzj2yv3z5xh7lks3hpr0000gn/T/ipykernel_58263/2878062256.py:3: SyntaxWarning: invalid escape sequence '\)'
  emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
/var/folders/_h/yzv4_kzj2yv3z5xh7lks3hpr0000gn/T/ipykernel_58263/2878062256.py:4: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub("[^\w']+",' ', text.lower()) + ' ' +' '.join(emoticons).replace('-','')


In [13]:
line, label = train_dataset[0]

In [14]:
line

"Just don't bother. I thought I would see a movie with great supspense and action.<br /><br />But it grows boring and terribly predictable after the interesting start. In the middle of the film you have a little social drama and all tension is lost because it slows down the speed. Towards the end the it gets better but not really great. I think the director took this movie just too serious. In such a kind of a movie even if u don't care about the plot at least you want some nice action. I nearly dozed off in the middle/main part of it. Rating 3/10.<br /><br />derboiler."

In [15]:
tokens = tokenizer(line)

In [16]:
tokens

['just',
 "don't",
 'bother',
 'i',
 'thought',
 'i',
 'would',
 'see',
 'a',
 'movie',
 'with',
 'great',
 'supspense',
 'and',
 'action',
 'but',
 'it',
 'grows',
 'boring',
 'and',
 'terribly',
 'predictable',
 'after',
 'the',
 'interesting',
 'start',
 'in',
 'the',
 'middle',
 'of',
 'the',
 'film',
 'you',
 'have',
 'a',
 'little',
 'social',
 'drama',
 'and',
 'all',
 'tension',
 'is',
 'lost',
 'because',
 'it',
 'slows',
 'down',
 'the',
 'speed',
 'towards',
 'the',
 'end',
 'the',
 'it',
 'gets',
 'better',
 'but',
 'not',
 'really',
 'great',
 'i',
 'think',
 'the',
 'director',
 'took',
 'this',
 'movie',
 'just',
 'too',
 'serious',
 'in',
 'such',
 'a',
 'kind',
 'of',
 'a',
 'movie',
 'even',
 'if',
 'u',
 "don't",
 'care',
 'about',
 'the',
 'plot',
 'at',
 'least',
 'you',
 'want',
 'some',
 'nice',
 'action',
 'i',
 'nearly',
 'dozed',
 'off',
 'in',
 'the',
 'middle',
 'main',
 'part',
 'of',
 'it',
 'rating',
 '3',
 '10',
 'derboiler']

In [17]:
# using the counter class from the collections library
Counter(tokens)

Counter({'the': 9,
         'i': 4,
         'a': 4,
         'it': 4,
         'movie': 3,
         'and': 3,
         'in': 3,
         'of': 3,
         'just': 2,
         "don't": 2,
         'great': 2,
         'action': 2,
         'but': 2,
         'middle': 2,
         'you': 2,
         'bother': 1,
         'thought': 1,
         'would': 1,
         'see': 1,
         'with': 1,
         'supspense': 1,
         'grows': 1,
         'boring': 1,
         'terribly': 1,
         'predictable': 1,
         'after': 1,
         'interesting': 1,
         'start': 1,
         'film': 1,
         'have': 1,
         'little': 1,
         'social': 1,
         'drama': 1,
         'all': 1,
         'tension': 1,
         'is': 1,
         'lost': 1,
         'because': 1,
         'slows': 1,
         'down': 1,
         'speed': 1,
         'towards': 1,
         'end': 1,
         'gets': 1,
         'better': 1,
         'not': 1,
         'really': 1,
         'think': 1,


Building the vocabulary:

In [18]:
token_counts = Counter()

for line, label in train_dataset:
    tokens = tokenizer(line)
    token_counts.update(tokens)

print('Vocab size: ', len(token_counts))

Vocab size:  80284


Next, we go to mapping each unique word to an integer. This can be done manually ising a python dictionary where the keys are the unique tokens (words) and the value associated with each is a unique integer.

In [20]:
class Vocabulary:
    def __init__(self, ordered_dict: OrderedDict, unk_index: int=None):
        # ordered dict maps token -> frequency (frequency is ignored after init)
        self.itos: List[str] = list(ordered_dict.keys()) # index -> token
        self.stoi: Dict[str,int] = {tok:idx for idx,tok in enumerate(self.itos)}
        self.unk_index = unk_index
        self.default_index = unk_index 
    
    def insert_token(self, token: str, index: int) -> None:
        """Insert a token at a specific index (shifts existing entries)"""
        if token in self.stoi:
            # token already exists -> remove old entry
            old_idx = self.stoi.pop(token)
            # shift everything after old_idx down
            for t, i in self.stoi.items():
                if i > old_idx:
                    self.stoi[t] = i-1
            self.itos = [t for t,_ in sorted(self.stoi.items(), key=lambda x: x[1])]

        # insert at the requested index
        self.itos.insert(index, token)
        self.stoi[token] = index
        # shift everything >= index up by 1
        for t,i in self.stoi.items():
            if i >= index and t!=token:
                self.stoi[t] = i+1
    
    def set_default_index(self, idx: int) -> None:
        self.default_index = idx
    
    # convenience methods 
    def __getitem__(self, token:str) -> int:
        return self.stoi.get(token, self.default_index)
    
    def __len__(self) -> int:
        return len(self.itos)
    
    def lookup_indices(self, tokens: List[str]) -> List[int]:
        return [self[t] for t in tokens]
    
    def lookup_tokens(self, indices:List[int]) -> List[str]:
        return [self.itos[i] for i in indices if i < len(self.itos)]

In [26]:
sorted_by_freq_tuples = sorted(token_counts.items(), key = lambda x: x[1], reverse=True)
ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = Vocabulary(ordered_dict)
vocab.insert_token("<pad>", 0)
vocab.insert_token("<unk>", 1)
vocab.set_default_index(1)

In [28]:
# demonstrate the operation of the vocab object:
print([vocab[token] for token in ['this','is','an','example']])

[11, 7, 32, 455]


we can now define the text pipeline function to transform each text accordingly

In [29]:
text_pipeline = lambda x : [vocab[token] for token in tokenizer(x)]

we wull generate batches of samples using DataLoader and pass the data processing pipelines declared previously to the argument collate_fn...we will wrap the text encoding into the collate_batch function.

collate_fn internals:

In [30]:
from torch.utils.data import DataLoader, Dataset
import torch
from torch.nn.utils.rnn import pad_sequence

In [31]:
class MyDataset(Dataset):
    def __init__(self):
        self.data = [i for i in range(10)] # pretending these are data points
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index]

In [32]:
dataset = MyDataset()
loader = DataLoader(dataset, batch_size=4, shuffle=False)

In [33]:
for batch in loader:
    print(batch)

tensor([0, 1, 2, 3])
tensor([4, 5, 6, 7])
tensor([8, 9])


Under the hood:
- when dataloader wants to form a batch, it calls the getitem method several times to get individual samples. then it passes that list of samples to a function called the collate function (collate_fn). this function determines how those samples are then combined into one batch
- by default, pytorch, uses the default_collate, which tries to stack everything into tensors automatically

- The collate function = how to combine samples into one batch 
  - the collate function defines what to do with a list of items from the dataset

- Default collate function:
 - converts lists of numbers into tensors
 - stacks them along a new batch dimension
 - works recursively for tuples and dictionaries

In [34]:
data = [
    (torch.tensor([1,2]), 0),
    (torch.tensor([3,4]), 1),
    (torch.tensor([5,6]), 0)
]

loader = DataLoader(data, batch_size=2)

In [35]:
for batch in loader:
    print(batch)

[tensor([[1, 2],
        [3, 4]]), tensor([0, 1])]
[tensor([[5, 6]]), tensor([0])]


Need for custom collate functions:
 - default stacking fails when samples are not the same shape or same type
 - for example when working with variable-length text sequences, images of different sizes, or more complex data structures

In [36]:
class VariableLengthDataset(Dataset):
    def __init__(self):
        self.data = [[1,2],[1,2,3,4],[1,2,3,4,5]]
    
    def __getitem__(self, index):
        return torch.tensor(self.data[index])
    
    def __len__(self):
        return len(self.data)

In [37]:
dataset = VariableLengthDataset()

for item in dataset:
    print(item)

tensor([1, 2])
tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4, 5])


In [38]:
loader = DataLoader(dataset, batch_size=2)

In [39]:
for batch in loader:
    print(batch)

RuntimeError: stack expects each tensor to be equal size, but got [2] at entry 0 and [4] at entry 1

tensors have different sizes, and therefore cannot be simply stacked.

Custom collate function:
 - we can define how to combine variable-lenth sequences - for instance, pad them to same length

In [40]:
def pad_collate(batch):
    # batch is a list of tensors of different lengths
    print(batch)
    padded = pad_sequence(batch, batch_first=True, padding_value=0)
    return padded

In [41]:
loader = DataLoader(dataset, batch_size=2, collate_fn = pad_collate)

In [42]:
for batch in loader:
    print(batch)

[tensor([1, 2]), tensor([1, 2, 3, 4])]
tensor([[1, 2, 0, 0],
        [1, 2, 3, 4]])
[tensor([1, 2, 3, 4, 5])]
tensor([[1, 2, 3, 4, 5]])


another collate function example:

In [47]:
data = [
    (torch.tensor([1,2,3]), 0),
    (torch.tensor([4,5]), 1),
    (torch.tensor([6]), 0)
]

def nlp_collate(batch):
    print(batch)
    sequences, labels = zip(*batch)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.tensor(labels)
    return padded, labels

In [48]:
loader = DataLoader(data, batch_size=2, collate_fn=nlp_collate)

In [49]:
for x,y in loader:
    print(x)
    print(y)
    break

[(tensor([1, 2, 3]), 0), (tensor([4, 5]), 1)]
tensor([[1, 2, 3],
        [4, 5, 0]])
tensor([0, 1])


In [45]:
seq, lab = zip(*ll)

In [46]:
seq

(tensor([1., 2., 3.]), tensor([4., 5.]))

returning back to the code

In [50]:
def collate_batch(batch):
    label_list, text_list, lengths = [], [], []

    for _text, _label in batch:
        label_list.append(_label)
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        lengths.append(processed_text.size(0))
    label_list = torch.tensor(label_list)
    lengths = torch.tensor(lengths)
    padded_text_list = nn.utils.rnn.pad_sequence(text_list, batch_first=True)
    return padded_text_list, label_list, lengths

In [51]:
# take a small batch
dataloader = DataLoader(train_dataset, batch_size=4, shuffle=False, collate_fn=collate_batch)

In [52]:
text_batch, label_batch, length_batch = next(iter(dataloader))

In [53]:
print(text_batch)

tensor([[   40,    89,  1415,    10,   195,    10,    59,    64,     4,    17,
            16,    81, 47043,     3,   209,    18,     9,  3382,   347,     3,
          1891,   728,    99,     2,   219,   372,     8,     2,   644,     5,
             2,    19,    22,    25,     4,   117,  1041,   453,     3,    29,
          1109,     7,   422,    85,     9,  8463,   178,     2,  2101,   951,
             2,   127,     2,     9,   213,   125,    18,    21,    63,    81,
            10,   101,     2,   164,   564,    11,    17,    40,    96,   633,
             8,   137,     4,   240,     5,     4,    17,    54,    45,  1214,
            89,   446,    41,     2,   110,    30,   216,    22,   177,    47,
           326,   209,    10,   752, 18164,   123,     8,     2,   644,   289,
           172,     5,     9,   677,   346,   160, 47044,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [67]:
print(label_batch)

tensor([0, 1, 0, 1])


In [68]:
print(length_batch)

tensor([107, 225, 248, 174])


In [54]:
text_batch.shape

torch.Size([4, 248])

In [55]:
length_batch

tensor([107, 225, 248, 174])

finally, let us divide all 3 datasets into data loaders with a batch size of 32

In [57]:
batch_size = 128
train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_dl = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

Embedding layers for sentence encoding

In [58]:
embedding = nn.Embedding(
    num_embeddings = 10,
    embedding_dim=3,
    padding_idx = 0
)

# a batch of 2 samples of 4 indices each
text_encoded_input = torch.LongTensor([[1,2,4,5],[4,3,2,0]])
print(embedding(text_encoded_input))

tensor([[[-1.2696, -1.7865, -1.5057],
         [ 0.4989,  0.4589,  0.3411],
         [ 1.1144, -1.2866,  1.6053],
         [ 0.8095, -0.1087,  1.5062]],

        [[ 1.1144, -1.2866,  1.6053],
         [-0.1597, -0.2262, -1.6432],
         [ 0.4989,  0.4589,  0.3411],
         [ 0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>)


**Building the RNN Model:**
- Now, we can build the RNN model, using the nn.Module class

In [60]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        _, hidden = self.rnn(x)
        out = hidden[-1, :, :] # we use the final hidden state from the last hidden layer as the input to the fc layer, as this is equal to the output og the rnn's last time step
        out = self.fc(out)
        return out

In [61]:
model = RNN(64, 32)
print(model)

RNN(
  (rnn): RNN(64, 32, num_layers=2, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


In [62]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

Building an RNN for the sentiment analysis task:

In [63]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_state, fc_hidden_state):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc1 = nn.Linear(rnn_hidden_size, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, text, lengths):
        out = self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )
        out, (hidden, cell) = self.rnn(out)
        out = hidden[-1, :, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

In [64]:
vocab_size = len(vocab)
embed_dim = 20
rnn_hidden_size = 64
fc_hidden_size = 64
torch.manual_seed(1)

In [65]:
model = RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size).to(device)

In [66]:
model

RNN(
  (embedding): Embedding(80286, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [69]:
def train(dataloader):
    model.train()
    total_acc, total_loss = 0, 0
    for text_batch, label_batch, lengths in dataloader:
        optimizer.zero_grad(set_to_none=True)
        label_batch = label_batch.float()
        text_batch, label_batch = text_batch.to(device), label_batch.to(device)
        pred = model(text_batch, lengths)[:, 0]
        loss = loss_fn(pred, label_batch)
        loss.backward()
        optimizer.step()
        total_acc += (
            (pred >= 0.5).float() == label_batch
        ).float().sum().item()

        total_loss += loss.item() * label_batch.size(0)
    return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)


def evaluate(dataloader):
    model.eval()
    total_acc, total_loss = 0, 0
    with torch.no_grad():
        for text_batch, label_batch, lengths in dataloader:
            label_batch = label_batch.float()
            text_batch, label_batch = text_batch.to(device), label_batch.to(device)
            pred = model(text_batch, lengths)[:,0]
            loss = loss_fn(pred, label_batch)
            total_acc += (
                (pred >= 0.5).float() == label_batch
            ).float().sum().item()
            total_loss += loss.item()*label_batch.size(0)
    return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [70]:
num_epochs = 10
torch.manual_seed(1)
for epoch in range(num_epochs):
    acc_train, loss_train = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)
    print(f'Epoch: {epoch} | accuracy: {acc_train:.4f} | val_accuracy: {acc_valid:.4f}')

Epoch: 0 | accuracy: 0.5565 | val_accuracy: 0.5942
Epoch: 1 | accuracy: 0.6600 | val_accuracy: 0.6484
Epoch: 2 | accuracy: 0.7246 | val_accuracy: 0.5828
Epoch: 3 | accuracy: 0.7239 | val_accuracy: 0.7164
Epoch: 4 | accuracy: 0.7852 | val_accuracy: 0.7348
Epoch: 5 | accuracy: 0.7933 | val_accuracy: 0.7826
Epoch: 6 | accuracy: 0.8435 | val_accuracy: 0.8018
Epoch: 7 | accuracy: 0.8697 | val_accuracy: 0.8154
Epoch: 8 | accuracy: 0.8880 | val_accuracy: 0.8142
Epoch: 9 | accuracy: 0.8961 | val_accuracy: 0.8302


**Character level language modelling in Pytorch:** 

In [71]:
with open('/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/data/1268-0.txt', 'r', encoding='utf-8') as fp:
    text = fp.read()

In [73]:
print(text)

*** START OF THE PROJECT GUTENBERG EBOOK 1268 ***

THE MYSTERIOUS ISLAND

by Jules Verne

1874




PART 1--DROPPED FROM THE CLOUDS



Chapter 1

“Are we rising again?” “No. On the contrary.” “Are we descending?”
 “Worse than that, captain! we are falling!” “For Heaven’s sake heave out
the ballast!” “There! the last sack is empty!” “Does the balloon rise?”
 “No!” “I hear a noise like the dashing of waves. The sea is below the
car! It cannot be more than 500 feet from us!” “Overboard with every
weight! ... everything!”

Such were the loud and startling words which resounded through the air,
above the vast watery desert of the Pacific, about four o’clock in the
evening of the 23rd of March, 1865.

Few can possibly have forgotten the terrible storm from the northeast,
in the middle of the equinox of that year. The tempest raged without
intermission from the 18th to the 26th of March. Its ravages were
terrible in America, Europe, and Asia, covering a distance of eighteen
hundred miles, and 

In [74]:
start_indx = text.find('THE MYSTERIOUS ISLAND')
end_indx = text.find('End of the Project Gutenberg')

In [75]:
start_indx, end_indx

(51, -1)

In [76]:
text = text[start_indx:end_indx]

char_set = set(text)

In [79]:
print('Total Length: ', len(text))
print('Unique characters: ', len(char_set))

Total Length:  1112310
Unique characters:  80


Need to map the characters to integers. To do this, we create a simple python dictionary that maps each character to an integer. char2int. we also need a reverse mapping to convert the results of our model back to text. 

In [81]:
import numpy as np

In [82]:
chars_sorted = sorted(char_set)
char2int = {ch:i for i,ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)

In [85]:
text_encoded = np.array(
    [char2int[ch] for ch in text],
    dtype = np.int32
)

text_encoded

array([44, 32, 29, ...,  6,  6,  6], dtype=int32)

In [86]:
print('Text encoded shape: ', text_encoded.shape)

Text encoded shape:  (1112310,)


In [87]:
print(text[:15], " == Encoding ==> ", text_encoded[:15])

THE MYSTERIOUS   == Encoding ==>  [44 32 29  1 37 48 43 44 29 42 33 39 45 43  1]


In [88]:
print(text_encoded[:15], " == Encoding ==> ", ''.join(char_array[text_encoded[:15]]))

[44 32 29  1 37 48 43 44 29 42 33 39 45 43  1]  == Encoding ==>  THE MYSTERIOUS 


so, text_encoded numpy array contains the encoded values for all characters in the text. Now, we will print out the mappings of the first 5 characters from this array

In [89]:
for ex in text_encoded[:5]:
    print('{} -> {}'.format(ex, char_array[ex]))

44 -> T
32 -> H
29 -> E
1 ->  
37 -> M


**formulating the problem:**
- Text generation can be phrased as a classification task. As in the image 

In [98]:
seq_length = 40
chunk_size = seq_length + 1
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded)-chunk_size + 1)]

In [99]:
text_chunks[:2]

[array([44, 32, 29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43,
        36, 25, 38, 28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54,
        67, 63, 54,  0,  0, 12, 19], dtype=int32),
 array([32, 29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43, 36,
        25, 38, 28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54, 67,
        63, 54,  0,  0, 12, 19, 18], dtype=int32)]

In [100]:
len(text_chunks)

1112270

In [101]:
class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks
    
    def __len__(self):
        return len(self.text_chunks)
    
    def __getitem__(self, index):
        text_chunk = self.text_chunks[index]
        return text_chunk[:-1].long(), text_chunk[1:].long()

In [102]:
seq_dataset = TextDataset(torch.tensor(text_chunks))

/var/folders/_h/yzv4_kzj2yv3z5xh7lks3hpr0000gn/T/ipykernel_58263/2835289213.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  seq_dataset = TextDataset(torch.tensor(text_chunks))


taking a look at some of the example sequences in the dataset

In [105]:
for i, (seq, target) in enumerate(seq_dataset):
    print(seq)
    print(target)
    break

tensor([44, 32, 29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43, 36,
        25, 38, 28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54, 67, 63,
        54,  0,  0, 12])
tensor([32, 29,  1, 37, 48, 43, 44, 29, 42, 33, 39, 45, 43,  1, 33, 43, 36, 25,
        38, 28,  0,  0, 51, 74,  1, 34, 70, 61, 54, 68,  1, 46, 54, 67, 63, 54,
         0,  0, 12, 19])


In [103]:
for i, (seq, target) in enumerate(seq_dataset):
    print(' Input(x): ',
    repr(''.join(char_array[seq])))
    print('Target (y): ', repr(''.join(char_array[target])))
    print()
    if i==1:
        break

 Input(x):  'THE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n1'
Target (y):  'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'

 Input(x):  'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'
Target (y):  'E MYSTERIOUS ISLAND\n\nby Jules Verne\n\n187'



finally, the last step: converting the data to minibatches

In [106]:
batch_size = 128
torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [107]:
sb, tb = next(iter(seq_dl))

In [108]:
sb.shape, tb.shape

(torch.Size([128, 40]), torch.Size([128, 40]))

In [110]:
z = sb[:, 1]
z.shape

torch.Size([128])

In [111]:
zl = nn.Embedding(vocab_size, 20)

In [113]:
zl(z).unsqueeze(1).shape

torch.Size([128, 1, 20])

In [114]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)
    
    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsqueeze(1)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden, cell
    
    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size)
        cell = torch.zeros(1, batch_size, self.rnn_hidden_size)
        return hidden, cell

In [115]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512
torch.manual_seed(1)

In [116]:
model = RNN(vocab_size, embed_dim, rnn_hidden_size)
model

RNN(
  (embedding): Embedding(80, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [117]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = model.to(device)

In [118]:
# next we have to create a loss function and an optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [119]:
num_epochs = 10000
torch.manual_seed(1)

data_iter = iter(seq_dl)

for epoch in range(num_epochs):
    hidden, cell = model.init_hidden(batch_size)
    hidden, cell = hidden.to(device), cell.to(device)
    seq_batch, target_batch = next(data_iter)
    seq_batch, target_batch = seq_batch.to(device), target_batch.to(device)
    optimizer.zero_grad()
    loss = 0 
    for c in range(seq_length):
        pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
        loss += loss_fn(pred, target_batch[:, c])
    loss.backward()
    optimizer.step()
    loss = loss.item()/seq_length
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | loss: {loss:.4f}")

Epoch 0 | loss: 4.3708
Epoch 500 | loss: 1.3027
Epoch 1000 | loss: 1.2229
Epoch 1500 | loss: 1.1956
Epoch 2000 | loss: 1.1531
Epoch 2500 | loss: 1.1197
Epoch 3000 | loss: 1.0737
Epoch 3500 | loss: 1.0729
Epoch 4000 | loss: 1.0695
Epoch 4500 | loss: 1.0868
Epoch 5000 | loss: 1.0505
Epoch 5500 | loss: 1.0808
Epoch 6000 | loss: 1.0893
Epoch 6500 | loss: 1.0683
Epoch 7000 | loss: 1.0622
Epoch 7500 | loss: 1.0666
Epoch 8000 | loss: 1.0858
Epoch 8500 | loss: 1.0611


StopIteration: 

Evaluating the model:
- **Generating new text passages**

In [120]:
from torch.distributions.categorical import Categorical

In [121]:
torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 1.0]])

In [122]:
print('Probabilities: ', nn.functional.softmax(logits, dim=1).numpy()[0])

Probabilities:  [0.33333334 0.33333334 0.33333334]


In [123]:
m = Categorical(logits = logits)
samples = m.sample((10,))
print(samples.numpy())

[[0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [2]
 [1]
 [1]]


In [125]:
torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 3.0]])
print('Probabilities: ', nn.functional.softmax(logits, dim=1).numpy()[0])
m = Categorical(logits = logits)
samples = m.sample((10,))
print(samples.numpy())

Probabilities:  [0.10650698 0.10650698 0.78698605]
[[0]
 [2]
 [2]
 [1]
 [2]
 [1]
 [2]
 [2]
 [2]
 [2]]


In [126]:
z = torch.randn(1, 40, 256)

In [128]:
z[:, 2].shape

torch.Size([1, 256])

In [132]:
def sample(model, starting_str, len_generated_text=500, scale_factor=1.0):
    encoded_input = torch.tensor(
        [char2int[s] for s in starting_str]
    )
    encoded_input = encoded_input.view(1, -1).to(device)
    generated_str = starting_str
    model.eval()
    hidden, cell = model.init_hidden(1)
    hidden, cell = hidden.to(device), cell.to(device)

    for c in range(len(starting_str)-1):
        _, hidden, cell = model(
            encoded_input[:, c].view(1), hidden, cell
        )
    last_char = encoded_input[:, -1]

    for i in range(len_generated_text):
        logits, hidden, cell = model(
            last_char.view(1), hidden, cell
        )
        logits = torch.squeeze(logits, 0)
        scaled_logits = logits * scale_factor
        m = Categorical(logits=scaled_logits)
        last_char = m.sample()
        generated_str += str(char_array[last_char])
    return generated_str

In [134]:
torch.manual_seed(1)
print(sample(model, starting_str='The island'))

The island was fanot for the lad like so many means of their steps, and in an idea of the Nopportunes, looking by Neb removably.

Cyrus Harding had the sandy at the struggle could say several being, by which fresh isn’t all human; but her is that?” asked the sailor, sea see their poper by firin. A thousand miles, and accumulled
on board, and found allowed, elecculing his care station shows of event moor, and an amission as he was on their heavy. Towards the shore, he climbed his
ascent to his two miles
fr


In [135]:
torch.manual_seed(1)
print(sample(model, starting_str='The island',scale_factor=2.0))

The island was finished, the lad will be discovered with the sea, which is an instant among the shore, and the balloon could be leaving their strength, and also the stones were good against the darkness, which might be supplied a shell of the gate of the base of the sea by some day, for the country appeared the beach, and the settlers were offered all the sea and storehouses by the colonists began to get up the sun, and as he was on their heads. They were therefore there to the corral, when he would be
su


In [137]:
torch.manual_seed(1)
print(sample(model, starting_str='The island',scale_factor=1.5))

The island was finished, the lad supplied themselves and did some place. Then the wind had been set at last, but the sea still lighted
the beach. The settlers succeeded in the river and fifty miles, the level of the northwest and speak for the sea hardened the interior of the Mercy, from the dog to the interior of the cliff, and a ball showed on their country, and that his
days after some days the “Bonadven our boat, we are more than twelve years of the charse of the crater; but, after turning to make bef


**RNN